# Assignment 2C — Part C: Prompt Router
Implements one-call classification, all five routes, the conditional few-shot classifier refinement, and router versus Zero-shot evaluation over the same benchmark questions.


In [1]:

from pathlib import Path
import subprocess
import os

REPO_URL = "https://github.com/tusharchouhan/banking-compliance-llm-assignment-1a.git"
PROJECT = Path("/content/banking-compliance-llm-assignment-1a")

if not (PROJECT / "src").exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT)], check=True)

os.chdir(PROJECT)
print("Project:", Path.cwd())
print("src exists:", (PROJECT / "src").exists())

Project: /content/banking-compliance-llm-assignment-1a
src exists: True


In [2]:
!ls

'Assignment Documentation'		     notebooks	        results
 Banking-compliance-llm-assignment-1a.docx   README.md	        run_pipeline.py
 data					     reports	        src
 models					     requirements.txt


In [3]:
!python -m pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 9.0 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of notebook to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.9/68.9 MB 58.3 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 996.6/996.6 kB 64.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 52.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 133.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 163.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 111.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.9/947.9 kB 60.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 120.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
# This cell installs the model inference, PEFT, embedding, and FAISS retrieval dependencies.%pip install -q transformers peft accelerate sentence-transformers faiss-cpu

In [5]:
# This cell loads the Assignment 1 model or a configurable Hugging Face fallback, reads the first ten benchmark examples, and builds a FAISS index over cleaned domain text.
from pathlib import Path
import os, json, time, re, warnings
import torch, numpy as np, pandas as pd, faiss
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from sentence_transformers import SentenceTransformer

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = Path("/content/LLM_Project_1_2")
DATA = ROOT / "data"
MODEL_ID = os.getenv("MODEL_ID", "HuggingFaceTB/SmolLM2-360M")
BASE_PATH = os.getenv("BASE_MODEL_DIR", str(ROOT / "models" / "base_model"))
ADAPTER_PATH = os.getenv("ADAPTER_DIR", str(ROOT / "models" / "adapter_A"))
if not Path(BASE_PATH, "config.json").exists():
    BASE_PATH = MODEL_ID
tokenizer = AutoTokenizer.from_pretrained(BASE_PATH)
model = AutoModelForCausalLM.from_pretrained(
    BASE_PATH,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None
)
adapter_loaded = Path(ADAPTER_PATH, "adapter_config.json").exists()
if adapter_loaded:
    model = PeftModel.from_pretrained(model, ADAPTER_PATH)
else:
    warnings.warn("Assignment 1 adapter not found; using base model. Set ADAPTER_DIR to your uploaded adapter for required final results.")
model.eval()

def generate(prompt, max_new_tokens=256):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    started = time.perf_counter()
    with torch.inference_mode():
        output = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    answer = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return answer, int(output.shape[-1]), time.perf_counter() - started

dataset_path = DATA / "instruction_dataset" / "instruction_dataset.jsonl"
if not dataset_path.exists():
    raise FileNotFoundError(f"Expected benchmark file: {dataset_path}")
benchmark = []
with dataset_path.open(encoding="utf-8") as stream:
    for line in stream:
        if line.strip():
            row = json.loads(line)
            benchmark.append({
                "query": row.get("query", row.get("instruction", row.get("question", ""))),
                "answer": row.get("answer", row.get("response", ""))
            })
        if len(benchmark) == 10:
            break
if len(benchmark) < 10:
    raise ValueError("The benchmark file must contain at least ten question-answer records.")

def read_chunks(folder=DATA / "cleaned_txt", chunk_words=220, overlap=40):
    chunks = []
    step = chunk_words - overlap
    for file in folder.rglob("*.txt"):
        words = file.read_text(encoding="utf-8", errors="ignore").split()
        for start in range(0, len(words), step):
            text = " ".join(words[start:start + chunk_words])
            if text:
                chunks.append({"source": str(file.relative_to(folder)), "text": text})
    if not chunks:
        raise FileNotFoundError(f"No cleaned .txt corpus found under {folder}")
    return chunks

chunks = read_chunks()
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
vectors = embedder.encode([item["text"] for item in chunks], normalize_embeddings=True, show_progress_bar=True).astype("float32")
faiss_index = faiss.IndexFlatIP(vectors.shape[1])
faiss_index.add(vectors)

def retrieve(query, k=2):
    query_vector = embedder.encode([query], normalize_embeddings=True).astype("float32")
    scores, ids = faiss_index.search(query_vector, min(k, len(chunks)))
    return [{"score": float(score), **chunks[idx]} for score, idx in zip(scores[0], ids[0]) if idx >= 0]

print(f"Questions: {len(benchmark)} | text chunks: {len(chunks)} | GPU: {torch.cuda.is_available()} | adapter loaded: {adapter_loaded}")



config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.66k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

... (17 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/314 [00:00<?, ?it/s]

Questions: 10 | text chunks: 10044 | GPU: True | adapter loaded: False


In [6]:
# This cell implements the four required base patterns with comparable answer, token, and latency outputs. RAG also returns its retrieved chunks for inspection.
def zero_shot(query):
    return generate(query)

def few_shot(query, examples=None, k=3):
    examples = (examples or benchmark)[:k]
    demonstrations = "\n\n".join(
        f"Question: {item['query']}\nAnswer: {item['answer']}" for item in examples
    )
    prompt = f"Use these worked examples as guidance.\n{demonstrations}\n\nQuestion: {query}\nAnswer:"
    return generate(prompt)

def rag(query):
    hits = retrieve(query, k=2)
    context = "\n\n".join(f"[{item['source']}] {item['text']}" for item in hits)
    prompt = f"Answer using only the evidence below. If it is insufficient, say so.\nEvidence:\n{context}\n\nQuestion: {query}\nAnswer:"
    result = generate(prompt)
    return result, hits

def self_critique(query):
    draft, tokens1, latency1 = generate(f"Answer the question accurately: {query}")
    critique, tokens2, latency2 = generate(
        f"Check the draft for errors, missing points, and unsupported claims.\nQuestion: {query}\nDraft: {draft}\nCritique:"
    )
    revised, tokens3, latency3 = generate(
        f"Revise the draft using the critique. Do not add unsupported claims.\nQuestion: {query}\nDraft: {draft}\nCritique: {critique}\nRevised answer:"
    )
    return revised, tokens1 + tokens2 + tokens3, latency1 + latency2 + latency3, draft, critique

def accuracy_score(answer, reference):
    # This overlap estimate is an aid only; manually review and assign the official 0–2 rubric.
    answer_words = set(re.findall(r"\w+", answer.lower()))
    reference_words = set(re.findall(r"\w+", reference.lower()))
    overlap = len(answer_words & reference_words) / max(1, len(reference_words))
    return 2 if overlap >= 0.65 else (1 if overlap >= 0.30 else 0)



In [7]:
# This cell defines the Verification and Socratic chains and tests them against RAG on five targeted banking/regulatory query types.
def verification_chain(query):
    hits = retrieve(query, k=2)
    evidence = "\n\n".join(f"[{h['source']}] {h['text']}" for h in hits)
    draft, t1, l1 = generate(f"Answer from the supplied evidence only.\nEvidence:\n{evidence}\nQuestion: {query}\nDraft:")
    critique, t2, l2 = generate(
        f"Critique domain accuracy against the evidence. Identify unsupported claims.\nEvidence:\n{evidence}\nQuestion: {query}\nDraft: {draft}\nCritique:"
    )
    revised, t3, l3 = generate(
        f"Revise using the evidence and critique; do not invent facts.\nEvidence:\n{evidence}\nDraft: {draft}\nCritique: {critique}\nFinal:"
    )
    return {"answer": revised, "draft": draft, "critique": critique, "observations": hits,
            "tokens": t1+t2+t3, "latency_sec": l1+l2+l3}

def socratic_chain(query):
    decomposition, t0, l0 = generate(
        f"Break the question into exactly two independent, answerable sub-questions. Return one per line.\nQuestion: {query}"
    )
    sub_questions = [line.strip(" -*0123456789.\t") for line in decomposition.splitlines() if line.strip()][:2]
    if len(sub_questions) < 2:
        sub_questions = [query, query]
    observations, total_tokens, total_latency = [], t0, l0
    for sub_question in sub_questions:
        (answer, tokens, latency), hits = rag(sub_question)
        observations.append({"sub_question": sub_question, "answer": answer, "retrieved": hits})
        total_tokens += tokens
        total_latency += latency
    synthesis, tokens, latency = generate(
        f"Answer the original question using only these per-question findings.\nQuestion: {query}\nFindings: {observations}\nFinal answer:"
    )
    return {"answer": synthesis, "sub_questions": sub_questions, "observations": observations,
            "tokens": total_tokens+tokens, "latency_sec": total_latency+latency}

targeted_queries = [
    ("Factual + verification", "According to the corpus, what is the required record-retention period for this compliance process?"),
    ("Compliance check", "What evidence should an analyst verify before concluding that a customer onboarding control is compliant?"),
    ("Multi-part process", "What steps should be followed for a suspicious transaction and which roles are responsible?"),
    ("Two regulations", "Compare the relevant RBI and SEBI requirements for reporting a suspected financial crime."),
    ("Ambiguous domain-specific", "Does the exception apply in this case, and what facts are needed to decide?")
]
chain_rows = []
for query_type, query in targeted_queries:
    verify = verification_chain(query)
    socratic = socratic_chain(query)
    (rag_answer, rag_tokens, rag_latency), rag_hits = rag(query)
    chain_rows.append({
        "query_type": query_type, "query": query,
        "verification_answer": verify["answer"], "verification_tokens": verify["tokens"],
        "verification_draft": verify["draft"], "verification_critique": verify["critique"],
        "socratic_answer": socratic["answer"], "socratic_tokens": socratic["tokens"],
        "socratic_subquestions": socratic["sub_questions"], "socratic_observations": socratic["observations"],
        "rag_answer": rag_answer, "rag_tokens": rag_tokens, "rag_retrieved": rag_hits,
        "human_best_choice": None, "human_accuracy_verification_0_2": None,
        "human_accuracy_socratic_0_2": None, "human_accuracy_rag_0_2": None
    })
part_b_results = pd.DataFrame(chain_rows)
display(part_b_results)



,query_type,query,verification_answer,verification_tokens,verification_draft,verification_critique,socratic_answer,socratic_tokens,socratic_subquestions,socratic_observations,rag_answer,rag_tokens,rag_retrieved,human_best_choice,human_accuracy_verification_0_2,human_accuracy_socratic_0_2,human_accuracy_rag_0_2
0,Factual + verification,"According to the corpus, what is the required ...",19.1.22. The record-keeping period shall be fi...,3469,19.1.1. The record-keeping period shall be fiv...,19.1.12. The record-keeping period shall be fi...,"Question: According to the corpus, what is the...",4058,"[Answer: 12 months, Question: What is the requ...","[{'sub_question': 'Answer: 12 months', 'answer...",The required record-retention period is 5 year...,908,"[{'score': 0.5692165493965149, 'source': 'IRDA...",None,None,None,None
1,Compliance check,What evidence should an analyst verify before ...,[SEBI/1769776024792.txt] • Verify that the cus...,3373,[SEBI/1769776024792.txt] • Obtain an understan...,[SEBI/1769776024792.txt] • Verify that the cus...,Question: What evidence should an analyst veri...,4694,[Answer: The customer onboarding control is co...,[{'sub_question': 'Answer: The customer onboar...,The customer onboarding control is a control t...,877,"[{'score': 0.4417921304702759, 'source': 'SEBI...",None,None,None,None
2,Multi-part process,What steps should be followed for a suspicious...,114\n\n[Page 41]\n\n[Page 42]\n\n[Page 43]\n\n...,3799,111\n\n[Page 14] 112 (b) Transactions involvin...,[RBI/MD123C0C6F7FF70E34F3DA5623B9E72AB3181.txt...,Question: What is the current status of the tr...,4954,[Question: What is the current status of the t...,[{'sub_question': 'Question: What is the curre...,The PSO shall monitor suspicious transactions ...,1017,"[{'score': 0.5240437984466553, 'source': 'RBI/...",None,None,None,None
3,Two regulations,Compare the relevant RBI and SEBI requirements...,1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1....,3670,1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1....,1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1....,Question: Compare the relevant RBI and SEBI re...,4278,[Question: What are the key differences betwee...,[{'sub_question': 'Question: What are the key ...,The SEBI and RBI have different requirements f...,974,"[{'score': 0.6679743528366089, 'source': 'RBI/...",None,None,None,None
4,Ambiguous domain-specific,"Does the exception apply in this case, and wha...",1786957172778.txt\n\n[SEBI/1786957172778.txt] ...,3462,1786957172778.txt\n\n[SEBI/1786957172778.txt] ...,[SEBI/1786957172778.txt] (9) the provisions of...,Question: Does the exception apply in this cas...,4001,"[The first question is:, Does the exception ap...","[{'sub_question': 'The first question is:', 'a...","The Board may, exempt any person or class of p...",905,"[{'score': 0.4312416911125183, 'source': 'SEBI...",None,None,None,None


In [8]:
# This cell implements the one-call label classifier, the assignment routing table, and a five-example classifier audit.
LABELS = ["factual", "formatted", "knowledge", "multi-part", "high-stakes"]

def classify_query(query, examples=None):
    examples = examples or []
    example_text = "\n".join(f"Query: {x['query']}\nLabel: {x['label']}" for x in examples)
    prompt = (
        "Classify the query with exactly one label from factual, formatted, knowledge, multi-part, high-stakes. "
        "Return only the label.\n" + (f"Examples:\n{example_text}\n" if examples else "") +
        f"Query: {query}\nLabel:"
    )
    raw, tokens, latency = generate(prompt, max_new_tokens=12)
    label = raw.strip().lower().split()[0].strip(".,:;") if raw.strip() else ""
    if label not in LABELS:
        label = "knowledge"
    return label, tokens, latency, raw

def route_query(query, classifier_examples=None):
    label, classifier_tokens, classifier_latency, raw_label = classify_query(query, classifier_examples)
    if label == "factual":
        answer, tokens, latency = zero_shot(query)
    elif label == "formatted":
        answer, tokens, latency = few_shot(query)
    elif label == "knowledge":
        (answer, tokens, latency), _ = rag(query)
    elif label == "multi-part":
        result = socratic_chain(query)
        answer, tokens, latency = result["answer"], result["tokens"], result["latency_sec"]
    else:
        result = verification_chain(query)
        answer, tokens, latency = result["answer"], result["tokens"], result["latency_sec"]
    return {"label": label, "raw_classifier_output": raw_label, "answer": answer,
            "tokens_including_classifier": classifier_tokens+tokens,
            "latency_including_classifier_sec": classifier_latency+latency}

classifier_cases = [
    ("Define a simple banking term.", "factual"),
    ("Give the onboarding process as numbered steps.", "formatted"),
    ("Find the applicable RBI rule in the corpus.", "knowledge"),
    ("Explain the process and its exception.", "multi-part"),
    ("Is this action compliant and safe to proceed?", "high-stakes")
]
classifier_rows = []
for query, expected in classifier_cases:
    predicted, tokens, latency, raw = classify_query(query)
    classifier_rows.append({"query": query, "expected_label": expected, "classifier_label": predicted,
                            "correct": predicted == expected, "raw_output": raw})
classifier_table = pd.DataFrame(classifier_rows)
display(classifier_table)
classifier_accuracy_before = int(classifier_table["correct"].sum())
print(f"Classifier accuracy before refinement: {classifier_accuracy_before}/5")

# If baseline classifier accuracy is below 3/5, add one domain-specific example per label and reevaluate.
classifier_examples = [
    {"query": "Define a banking term in one sentence.", "label": "factual"},
    {"query": "List the steps in a compliance workflow.", "label": "formatted"},
    {"query": "Locate the RBI requirement for KYC records.", "label": "knowledge"},
    {"query": "Compare the process and its reporting exception.", "label": "multi-part"},
    {"query": "Decide whether this transaction should proceed under policy.", "label": "high-stakes"}
]
if classifier_accuracy_before < 3:
    refined_rows = []
    for query, expected in classifier_cases:
        predicted, _, _, raw = classify_query(query, classifier_examples)
        refined_rows.append({"query": query, "expected_label": expected, "classifier_label": predicted,
                             "correct": predicted == expected, "raw_output": raw})
    refined_table = pd.DataFrame(refined_rows)
    display(refined_table)
    print(f"Classifier accuracy after refinement: {int(refined_table['correct'].sum())}/5")

# This cell compares the router with always-zero-shot on the same ten questions; classifier tokens are included.
router_rows = []
for item in benchmark:
    routed = route_query(item["query"], classifier_examples if classifier_accuracy_before < 3 else None)
    direct_answer, direct_tokens, direct_latency = zero_shot(item["query"])
    router_rows.append({
        "query": item["query"], "route": routed["label"],
        "router_answer": routed["answer"], "router_tokens": routed["tokens_including_classifier"],
        "router_latency_sec": routed["latency_including_classifier_sec"],
        "zero_shot_answer": direct_answer, "zero_shot_tokens": direct_tokens,
        "zero_shot_latency_sec": direct_latency, "reference_answer": item["answer"],
        "router_preliminary_score": accuracy_score(routed["answer"], item["answer"]),
        "zero_shot_preliminary_score": accuracy_score(direct_answer, item["answer"]),
        "human_router_score_0_2": None, "human_zero_shot_score_0_2": None
    })
router_results = pd.DataFrame(router_rows)
display(router_results)
router_summary = pd.DataFrame([
    {"metric":"Total accuracy sum (0-2)","always_zero_shot":router_results.zero_shot_preliminary_score.sum(),"prompt_router":router_results.router_preliminary_score.sum()},
    {"metric":"Average tokens per query","always_zero_shot":router_results.zero_shot_tokens.mean(),"prompt_router":router_results.router_tokens.mean()},
    {"metric":"Classifier accuracy (of 5)","always_zero_shot":"—","prompt_router":f"{classifier_accuracy_before}/5"}
])
display(router_summary)



,query,expected_label,classifier_label,correct,raw_output
0,Define a simple banking term.,factual,knowledge,False,Banking\n\nQuery: Define a simple banking term.
1,Give the onboarding process as numbered steps.,formatted,knowledge,False,Onboarding\n\nQuery: Give the onboarding proce...
2,Find the applicable RBI rule in the corpus.,knowledge,knowledge,True,Find the RBI rule in the corpus.\n\n##
3,Explain the process and its exception.,multi-part,knowledge,False,Explain the process and its exception.\n\nThe ...
4,Is this action compliant and safe to proceed?,high-stakes,knowledge,False,Safe\n\nQuery: Is this action compliant and sa...


Classifier accuracy before refinement: 1/5


,query,expected_label,classifier_label,correct,raw_output
0,Define a simple banking term.,factual,knowledge,False,low-stakes\nQuery: List the steps in a compliance
1,Give the onboarding process as numbered steps.,formatted,high-stakes,False,high-stakes\nQuery: List the steps in a compli...
2,Find the applicable RBI rule in the corpus.,knowledge,high-stakes,False,high-stakes\nQuery: Find the RBI rule for
3,Explain the process and its exception.,multi-part,high-stakes,False,high-stakes\nQuery: List the steps in a compli...
4,Is this action compliant and safe to proceed?,high-stakes,high-stakes,True,high-stakes\nQuery: Is this transaction compli...


Classifier accuracy after refinement: 1/5


,query,route,router_answer,router_tokens,router_latency_sec,zero_shot_answer,zero_shot_tokens,zero_shot_latency_sec,reference_answer,router_preliminary_score,zero_shot_preliminary_score,human_router_score_0_2,human_zero_shot_score_0_2
0,What should a compliance analyst know about vi...,multi-part,Question: What should a compliance analyst kno...,4415,41.854086,The answer is that a compliance analyst should...,276,10.099223,"(vi) who, neither himself or herself, nor whos...",0,0,None,None
1,Summarize the compliance requirement described...,multi-part,Question: Summarize the compliance requirement...,5325,42.229192,The following is a summary of the compliance r...,277,10.188123,"Prior to its omission, it read as “Investor”.",2,2,None,None
2,What should a compliance analyst know about Th...,multi-part,Question: What should a compliance analyst kno...,4984,41.962340,a. The number of employees who are affected by...,276,10.192461,The following details regarding the issue to b...,1,0,None,None
3,Which banking or regulatory rule is described ...,knowledge,[SEBI/1786946157192.txt] The Securities and Ex...,845,10.792292,A.,277,9.996908,"64 dated February 4, 2004 r ead with GoI Notif...",2,0,None,None
4,What does the source document state about Page...,high-stakes,669[The words “i ndividual investors” shall be...,3907,31.511249,A. The source document states that Page invest...,274,10.168627,[Page 258] 258 investors” shall be read as ...,1,0,None,None
5,Which banking or regulatory rule is described ...,high-stakes,1999-01-01\n\nPage 9 of 110\n\nPage 9 of 110\n...,3556,31.131547,A. The insurance company shall convey the insu...,275,10.064442,"Where underwriting is done, the insurance...",0,0,None,None
6,Summarize the compliance requirement described...,high-stakes,1785910555405.txt\n\n[SEBI/1785910555405.txt] ...,3902,31.278727,10. 11. 12. 13. 14. 15. 16. 17. 18. 19. 20. 21...,276,10.133942,Chief Financial Officer and compliance officer...,0,0,None,None
7,What does the source document state about Any ...,high-stakes,[SEBI/1776685013526.txt] of business: a) date ...,3872,31.222327,The source document states that the Any Other ...,270,10.237416,Any Other Capacity (to be specified) ...,2,2,None,None
8,Summarize the compliance requirement described...,high-stakes,[SEBI/1785306756282.txt]\n\n[SEBI/178530675628...,4203,31.418925,Answer: The Securities and Exchange Board of I...,278,9.976821,20 Substituted vide Securities and Exchange Bo...,0,1,None,None
9,What does the source document state about l Fa...,high-stakes,[SEBI/1786079544836.txt]\n\n[SEBI/178607954483...,3595,31.127773,The source document states that the report is ...,274,10.079261,(l) Fairness report as mentioned in para 2(c) ...,0,0,None,None


,metric,always_zero_shot,prompt_router
0,Total accuracy sum (0-2),5,8
1,Average tokens per query,275.3,3860.4
2,Classifier accuracy (of 5),—,1/5


## Required 100-word analysis
Use human-reviewed scores to discuss whether routing improved accuracy, whether its extra classifier/pattern calls raised token cost, and whether the dominant failure was misclassification or a weak routed pattern.
